
# Publication graphs for the MobileNetV1–Fourier-INR experiment
Reads existing results only. It does not train or modify any model.



In [1]:

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------
RESULTS_DIR = Path(
    r"D:\imagecondition revision\MobileNetV1_ABCD_PatientLevel_10Fold_6Class_Results"
)
FINAL_MODEL_DIR = RESULTS_DIR / "final_model"
FOLDS_DIR = RESULTS_DIR / "folds"
GRAPH_ROOT = RESULTS_DIR / "Publication_Graphs_Model_B_Proposed"

GRAPH_DIRS = {
    "dataset": GRAPH_ROOT / "01_Dataset_and_Patient_Splits",
    "cv": GRAPH_ROOT / "02_Ten_Fold_Cross_Validation",
    "final": GRAPH_ROOT / "03_Final_Untouched_Test",
    "learning": GRAPH_ROOT / "04_Learning_Curves",
    "confusion": GRAPH_ROOT / "05_Confusion_Matrices",
    "classwise": GRAPH_ROOT / "06_Classwise_Performance",
    "efficiency": GRAPH_ROOT / "07_Computational_Efficiency",
    "statistics": GRAPH_ROOT / "08_Statistical_Analysis",
}
for directory in GRAPH_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

# Internal keys are unchanged for traceability. Proposed Model B is displayed last.
DISPLAY_ORDER = ["A", "C", "D", "B"]
DISPLAY_LABELS = {
    "A": "MobileNetV1 baseline",
    "C": "Image-conditioned Fourier-INR variant",
    "D": "Image-conditioned INR + cross-attention variant",
    "B": "Proposed Fourier-INR fusion",
}
SHORT_LABELS = {
    "A": "A: Baseline",
    "C": "C: Image-conditioned",
    "D": "D: Cross-attention",
    "B": "B: Proposed",
}
COLORS = {
    "A": "#7A7A7A",
    "C": "#56B4E9",
    "D": "#CC79A7",
    "B": "#0072B2",
}
CLASS_ORDER = ["Abdomen", "Brain", "Femur", "Maternal Cervix", "Thorax", "Other"]
SPLIT_ORDER = ["train", "val", "test"]
SPLIT_LABELS = {"train": "Training", "val": "Validation", "test": "Testing"}
SPLIT_COLORS = {"train": "#0072B2", "val": "#E69F00", "test": "#009E73"}

# ---------------------------------------------------------------------
# PUBLICATION STYLE
# ---------------------------------------------------------------------
sns.set_theme(style="whitegrid", context="paper")
mpl.rcParams.update({
    "figure.constrained_layout.use": True,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.titlesize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.18,
})

figure_manifest = []
skipped_figures = []


def save_figure(fig, folder_key, filename, title):
    """Save every figure as high-resolution PNG, vector PDF and SVG."""
    output_dir = GRAPH_DIRS[folder_key]
    for extension in ("png", "pdf", "svg"):
        path = output_dir / f"{filename}.{extension}"
        kwargs = {"dpi": 600} if extension == "png" else {}
        fig.savefig(path, **kwargs)
    figure_manifest.append({
        "Section": folder_key,
        "Figure": filename,
        "Title": title,
        "PNG": str(output_dir / f"{filename}.png"),
        "PDF": str(output_dir / f"{filename}.pdf"),
        "SVG": str(output_dir / f"{filename}.svg"),
    })
    plt.close(fig)


def require_file(filename):
    path = RESULTS_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Required results file not found: {path}")
    return path


def percent_axis(axis, which="y"):
    formatter = mpl.ticker.PercentFormatter(xmax=1.0, decimals=0)
    if which == "y":
        axis.yaxis.set_major_formatter(formatter)
    else:
        axis.xaxis.set_major_formatter(formatter)


def add_bar_labels(axis, decimals=1, percentage=False):
    for container in axis.containers:
        labels = []
        for bar in container:
            value = bar.get_height()
            if not np.isfinite(value):
                labels.append("")
            elif percentage:
                labels.append(f"{100 * value:.{decimals}f}%")
            else:
                labels.append(f"{value:.{decimals}f}")
        axis.bar_label(container, labels=labels, padding=2, fontsize=8)


def display_name(key):
    return DISPLAY_LABELS.get(str(key), str(key))


# ---------------------------------------------------------------------
# LOAD AUTHORITATIVE RESULTS
# ---------------------------------------------------------------------
fold_results = pd.read_csv(require_file("ABCD_10Fold_All_Results.csv"))
epoch_timings = pd.read_csv(require_file("ABCD_All_Epoch_Timings.csv"))
final_results = pd.read_csv(require_file("ABCD_Final_Untouched_Test_Results.csv"))
architecture = pd.read_csv(require_file("ABCD_Architecture_Parameters.csv"))
computational = pd.read_csv(require_file("ABCD_Computational_Comparison.csv"))
class_distribution = pd.read_csv(require_file("Classwise_Distribution.csv"))
dataset_manifest = pd.read_csv(require_file("Dataset_Manifest_with_Patient_IDs.csv"))
fold_assignments = pd.read_csv(require_file("Shared_Patient_Fold_Assignments.csv"))

friedman_path = RESULTS_DIR / "ABCD_Friedman_Tests.csv"
holm_path = RESULTS_DIR / "ABCD_Holm_Paired_Comparisons.csv"
friedman = pd.read_csv(friedman_path) if friedman_path.exists() else None
holm = pd.read_csv(holm_path) if holm_path.exists() else None

for frame in (fold_results, final_results, architecture, computational):
    key_column = "Model Key" if "Model Key" in frame.columns else "Key"
    frame["Display Model"] = frame[key_column].astype(str).map(DISPLAY_LABELS)
    frame["Display Model"] = pd.Categorical(
        frame["Display Model"],
        categories=[DISPLAY_LABELS[k] for k in DISPLAY_ORDER],
        ordered=True,
    )

print(f"Results loaded from: {RESULTS_DIR}")
print(f"Graphs will be saved to: {GRAPH_ROOT}")



Results loaded from: D:\imagecondition revision\MobileNetV1_ABCD_PatientLevel_10Fold_6Class_Results
Graphs will be saved to: D:\imagecondition revision\MobileNetV1_ABCD_PatientLevel_10Fold_6Class_Results\Publication_Graphs_Model_B_Proposed



## 1. Dataset composition and patient-level separation



In [2]:

# Figure 1: images per class and split
distribution_long = class_distribution.melt(
    id_vars="class_name",
    value_vars=[c for c in SPLIT_ORDER if c in class_distribution.columns],
    var_name="Split",
    value_name="Images",
)
distribution_long["Split Label"] = distribution_long["Split"].map(SPLIT_LABELS)

fig, ax = plt.subplots(figsize=(9.2, 5.3))
sns.barplot(
    data=distribution_long,
    x="class_name",
    y="Images",
    hue="Split Label",
    hue_order=[SPLIT_LABELS[s] for s in SPLIT_ORDER],
    palette=[SPLIT_COLORS[s] for s in SPLIT_ORDER],
    ax=ax,
)
ax.set_title("Image distribution across training, validation and testing sets")
ax.set_xlabel("Ultrasound class")
ax.set_ylabel("Number of images")
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Dataset split", frameon=False)
add_bar_labels(ax, decimals=0)
# Layout handled globally by constrained_layout.
save_figure(fig, "dataset", "Fig_01_Classwise_Image_Distribution", "Class-wise image distribution")

# Figure 2: total images per split
split_image_counts = (
    dataset_manifest.groupby("split", observed=False).size().reindex(SPLIT_ORDER).fillna(0)
)
fig, ax = plt.subplots(figsize=(6.5, 4.7))
bars = ax.bar(
    [SPLIT_LABELS[s] for s in SPLIT_ORDER],
    split_image_counts.values,
    color=[SPLIT_COLORS[s] for s in SPLIT_ORDER],
)
ax.bar_label(bars, labels=[f"{int(v):,}" for v in split_image_counts.values], padding=3)
ax.set_title("Number of images in each dataset split")
ax.set_xlabel("Dataset split")
ax.set_ylabel("Number of images")
# Layout handled globally by constrained_layout.
save_figure(fig, "dataset", "Fig_02_Images_Per_Split", "Images per dataset split")

# Figure 3: unique patients per split
split_patient_counts = (
    dataset_manifest.groupby("split", observed=False)["patient_id"].nunique()
    .reindex(SPLIT_ORDER).fillna(0)
)
fig, ax = plt.subplots(figsize=(6.5, 4.7))
bars = ax.bar(
    [SPLIT_LABELS[s] for s in SPLIT_ORDER],
    split_patient_counts.values,
    color=[SPLIT_COLORS[s] for s in SPLIT_ORDER],
)
ax.bar_label(bars, labels=[f"{int(v):,}" for v in split_patient_counts.values], padding=3)
ax.set_title("Patient-level distribution across dataset splits")
ax.set_xlabel("Dataset split")
ax.set_ylabel("Unique patients")
# Layout handled globally by constrained_layout.
save_figure(fig, "dataset", "Fig_03_Patients_Per_Split", "Unique patients per dataset split")

# Figure 4: split composition percentages
split_percent = split_image_counts / split_image_counts.sum()
fig, ax = plt.subplots(figsize=(7.0, 2.5))
left = 0.0
for split in SPLIT_ORDER:
    value = float(split_percent.loc[split])
    ax.barh(["All images"], [value], left=left, color=SPLIT_COLORS[split], label=SPLIT_LABELS[split])
    ax.text(left + value / 2, 0, f"{100 * value:.1f}%", ha="center", va="center", color="white", fontweight="bold")
    left += value
ax.set_xlim(0, 1)
percent_axis(ax, "x")
ax.set_title("Overall train–validation–test image allocation")
ax.set_xlabel("Percentage of all images")
ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.55), frameon=False)
# Layout handled globally by constrained_layout.
save_figure(fig, "dataset", "Fig_04_Split_Allocation_Percentage", "Train-validation-test allocation")

# Figure 5: class proportions within each split
class_split_table = pd.crosstab(dataset_manifest["class_name"], dataset_manifest["split"])
class_split_table = class_split_table.reindex(index=CLASS_ORDER, columns=SPLIT_ORDER).fillna(0)
class_split_percent = class_split_table.div(class_split_table.sum(axis=0), axis=1)
fig, ax = plt.subplots(figsize=(7.5, 5.3))
sns.heatmap(
    class_split_percent * 100,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    cbar_kws={"label": "Percentage within split (%)"},
    ax=ax,
)
ax.set_title("Class composition within each dataset split")
ax.set_xlabel("Dataset split")
ax.set_ylabel("Ultrasound class")
ax.set_xticklabels([SPLIT_LABELS[s] for s in SPLIT_ORDER], rotation=0)
# Layout handled globally by constrained_layout.
save_figure(fig, "dataset", "Fig_05_Class_Proportion_Heatmap", "Class proportions within splits")

# Figure 6: explicit patient overlap matrix (diagonal = patients; off-diagonal should be zero)
patient_sets = {
    split: set(dataset_manifest.loc[dataset_manifest["split"] == split, "patient_id"].astype(str))
    for split in SPLIT_ORDER
}
overlap = pd.DataFrame(index=SPLIT_ORDER, columns=SPLIT_ORDER, dtype=int)
for row_split in SPLIT_ORDER:
    for column_split in SPLIT_ORDER:
        overlap.loc[row_split, column_split] = len(patient_sets[row_split] & patient_sets[column_split])
fig, ax = plt.subplots(figsize=(5.8, 4.8))
sns.heatmap(overlap.astype(int), annot=True, fmt="d", cmap="Greens", cbar=False, linewidths=1, ax=ax)
ax.set_title("Patient-overlap verification between dataset splits")
ax.set_xlabel("Dataset split")
ax.set_ylabel("Dataset split")
ax.set_xticklabels([SPLIT_LABELS[s] for s in SPLIT_ORDER], rotation=0)
ax.set_yticklabels([SPLIT_LABELS[s] for s in SPLIT_ORDER], rotation=0)
# Layout handled globally by constrained_layout.
save_figure(fig, "dataset", "Fig_06_Patient_Overlap_Matrix", "Patient overlap verification")

# Figure 7: fold roles by patient count
fold_role_counts = (
    fold_assignments.groupby(["Fold", "Role"])["Patient ID"].nunique().reset_index(name="Patients")
)
role_order = ["fit", "inner_validation", "outer_test"]
role_labels = {"fit": "Model fitting", "inner_validation": "Early-stopping validation", "outer_test": "Outer-fold evaluation"}
fig, ax = plt.subplots(figsize=(9.0, 5.1))
sns.barplot(
    data=fold_role_counts,
    x="Fold",
    y="Patients",
    hue="Role",
    hue_order=role_order,
    palette=["#0072B2", "#E69F00", "#009E73"],
    ax=ax,
)
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, [role_labels[r] for r in role_order], title="Fold role", frameon=False)
ax.set_title("Patient allocation within each outer cross-validation fold")
ax.set_xlabel("Outer fold")
ax.set_ylabel("Unique patients")
# Layout handled globally by constrained_layout.
save_figure(fig, "dataset", "Fig_07_Patient_Roles_Across_10_Folds", "Patient roles across folds")

# Figure 8: outer-test images and patients per fold (same for every model; use Model A once)
fold_sizes = fold_results.loc[fold_results["Model Key"] == "A", ["Fold", "Outer Test Images", "Outer Test Patients"]].copy()
fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.4))
axes[0].bar(fold_sizes["Fold"], fold_sizes["Outer Test Images"], color="#56B4E9")
axes[0].set_title("Outer-test images per fold")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("Images")
axes[0].set_xticks(range(1, 11))
axes[1].bar(fold_sizes["Fold"], fold_sizes["Outer Test Patients"], color="#009E73")
axes[1].set_title("Outer-test patients per fold")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("Patients")
axes[1].set_xticks(range(1, 11))
fig.suptitle("Size of the locked outer evaluation partitions")
# Layout handled globally by constrained_layout.
save_figure(fig, "dataset", "Fig_08_Outer_Fold_Sizes", "Outer-fold image and patient counts")




## 2. Ten-fold cross-validation results



In [3]:

METRICS = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Kappa", "MCC"]

# Figure 9: fold-wise accuracy lines
fig, ax = plt.subplots(figsize=(9.2, 5.2))
for key in DISPLAY_ORDER:
    data = fold_results[fold_results["Model Key"] == key].sort_values("Fold")
    ax.plot(
        data["Fold"], data["Accuracy"], marker="o", linewidth=2.4 if key == "B" else 1.5,
        markersize=6 if key == "B" else 4, color=COLORS[key], label=SHORT_LABELS[key],
    )
ax.set_xticks(range(1, 11))
ax.set_ylim(0.84, 0.94)
percent_axis(ax)
ax.set_title("Fold-wise accuracy under identical patient-level partitions")
ax.set_xlabel("Outer fold")
ax.set_ylabel("Accuracy")
ax.legend(frameon=False, ncol=2)
# Layout handled globally by constrained_layout.
save_figure(fig, "cv", "Fig_09_Foldwise_Accuracy_All_Models", "Fold-wise accuracy comparison")

# Figure 10: fold-wise Macro F1 lines
fig, ax = plt.subplots(figsize=(9.2, 5.2))
for key in DISPLAY_ORDER:
    data = fold_results[fold_results["Model Key"] == key].sort_values("Fold")
    ax.plot(data["Fold"], data["Macro F1"], marker="o", linewidth=2.4 if key == "B" else 1.5,
            markersize=6 if key == "B" else 4, color=COLORS[key], label=SHORT_LABELS[key])
ax.set_xticks(range(1, 11))
percent_axis(ax)
ax.set_title("Fold-wise macro F1-score under identical patient-level partitions")
ax.set_xlabel("Outer fold")
ax.set_ylabel("Macro F1-score")
ax.legend(frameon=False, ncol=2)
# Layout handled globally by constrained_layout.
save_figure(fig, "cv", "Fig_10_Foldwise_Macro_F1_All_Models", "Fold-wise macro F1 comparison")

# Figure 11: accuracy boxplot with individual folds
fig, ax = plt.subplots(figsize=(9.0, 5.2))
order_names = [DISPLAY_LABELS[k] for k in DISPLAY_ORDER]
sns.boxplot(data=fold_results, x="Display Model", y="Accuracy", order=order_names,
            palette=[COLORS[k] for k in DISPLAY_ORDER], width=0.6, showfliers=False, ax=ax)
sns.stripplot(data=fold_results, x="Display Model", y="Accuracy", order=order_names,
              color="black", size=4, jitter=0.12, alpha=0.65, ax=ax)
percent_axis(ax)
ax.set_title("Distribution of accuracy across the ten patient-level folds")
ax.set_xlabel("")
ax.set_ylabel("Accuracy")
ax.tick_params(axis="x", rotation=18)
# Layout handled globally by constrained_layout.
save_figure(fig, "cv", "Fig_11_Accuracy_Boxplot_10_Folds", "Ten-fold accuracy distributions")

# Figure 12: macro F1 boxplot
fig, ax = plt.subplots(figsize=(9.0, 5.2))
sns.boxplot(data=fold_results, x="Display Model", y="Macro F1", order=order_names,
            palette=[COLORS[k] for k in DISPLAY_ORDER], width=0.6, showfliers=False, ax=ax)
sns.stripplot(data=fold_results, x="Display Model", y="Macro F1", order=order_names,
              color="black", size=4, jitter=0.12, alpha=0.65, ax=ax)
percent_axis(ax)
ax.set_title("Distribution of macro F1-score across the ten folds")
ax.set_xlabel("")
ax.set_ylabel("Macro F1-score")
ax.tick_params(axis="x", rotation=18)
# Layout handled globally by constrained_layout.
save_figure(fig, "cv", "Fig_12_Macro_F1_Boxplot_10_Folds", "Ten-fold macro F1 distributions")

# Figure 13: mean ± SD for all metrics
summary_rows = []
for key in DISPLAY_ORDER:
    subset = fold_results[fold_results["Model Key"] == key]
    for metric in METRICS:
        summary_rows.append({
            "Model Key": key,
            "Model": SHORT_LABELS[key],
            "Metric": metric,
            "Mean": subset[metric].mean(),
            "SD": subset[metric].std(ddof=1),
        })
cv_summary = pd.DataFrame(summary_rows)
fig, ax = plt.subplots(figsize=(10.0, 5.6))
x = np.arange(len(METRICS))
width = 0.19
for index, key in enumerate(DISPLAY_ORDER):
    subset = cv_summary[cv_summary["Model Key"] == key].set_index("Metric").reindex(METRICS)
    position = x + (index - 1.5) * width
    ax.bar(position, subset["Mean"], width, yerr=subset["SD"], capsize=3,
           color=COLORS[key], label=SHORT_LABELS[key], alpha=0.95)
ax.set_xticks(x)
ax.set_xticklabels(METRICS, rotation=15, ha="right")
ax.set_ylim(0.80, 0.94)
percent_axis(ax)
ax.set_title("Ten-fold performance comparison (mean ± standard deviation)")
ax.set_xlabel("Evaluation metric")
ax.set_ylabel("Mean score")
ax.legend(frameon=False, ncol=2)
# Layout handled globally by constrained_layout.
save_figure(fig, "cv", "Fig_13_All_Metrics_Mean_SD", "Ten-fold mean and standard deviation")

# Figure 14: heatmap of fold-wise accuracy
accuracy_heatmap = fold_results.pivot(index="Model Key", columns="Fold", values="Accuracy").reindex(DISPLAY_ORDER)
fig, ax = plt.subplots(figsize=(10.0, 4.0))
sns.heatmap(accuracy_heatmap * 100, annot=True, fmt=".1f", cmap="YlGnBu",
            cbar_kws={"label": "Accuracy (%)"}, linewidths=0.4, ax=ax)
ax.set_yticklabels([SHORT_LABELS[k] for k in DISPLAY_ORDER], rotation=0)
ax.set_title("Accuracy of every model in every patient-level fold")
ax.set_xlabel("Outer fold")
ax.set_ylabel("")
# Layout handled globally by constrained_layout.
save_figure(fig, "cv", "Fig_14_Foldwise_Accuracy_Heatmap", "Fold-wise accuracy heatmap")

# Figure 15: rank heatmap (1 = best within the fold)
rank_heatmap = accuracy_heatmap.rank(axis=0, ascending=False, method="average")
fig, ax = plt.subplots(figsize=(10.0, 4.0))
sns.heatmap(rank_heatmap, annot=True, fmt=".1f", cmap="RdYlGn_r", vmin=1, vmax=4,
            cbar_kws={"label": "Accuracy rank (1 = best)"}, linewidths=0.4, ax=ax)
ax.set_yticklabels([SHORT_LABELS[k] for k in DISPLAY_ORDER], rotation=0)
ax.set_title("Within-fold model ranks based on accuracy")
ax.set_xlabel("Outer fold")
ax.set_ylabel("")
# Layout handled globally by constrained_layout.
save_figure(fig, "cv", "Fig_15_Within_Fold_Accuracy_Ranks", "Within-fold accuracy ranks")

# Figure 16: paired A-to-B accuracy changes
paired = fold_results.pivot(index="Fold", columns="Model Key", values="Accuracy")
fig, ax = plt.subplots(figsize=(7.5, 5.3))
for fold, row in paired.iterrows():
    ax.plot([0, 1], [row["A"], row["B"]], color="#999999", alpha=0.65, linewidth=1)
    ax.scatter(0, row["A"], color=COLORS["A"], s=35)
    ax.scatter(1, row["B"], color=COLORS["B"], s=45)
ax.set_xticks([0, 1])
ax.set_xticklabels(["MobileNetV1 baseline", "Proposed Fourier-INR fusion"])
percent_axis(ax)
ax.set_title("Paired fold-wise accuracy: baseline versus proposed model")
ax.set_ylabel("Accuracy")
ax.set_xlabel("")
# Layout handled globally by constrained_layout.
save_figure(fig, "cv", "Fig_16_Paired_A_vs_B_Accuracy", "Paired baseline-proposed accuracy")

# Figure 17: B-A accuracy difference per fold
delta = paired["B"] - paired["A"]
fig, ax = plt.subplots(figsize=(8.5, 4.8))
delta_colors = [COLORS["B"] if value >= 0 else "#D55E00" for value in delta]
bars = ax.bar(delta.index.astype(str), delta.values, color=delta_colors)
ax.axhline(0, color="black", linewidth=1)
ax.axhline(delta.mean(), color=COLORS["B"], linestyle="--", linewidth=1.5,
           label=f"Mean difference = {100 * delta.mean():.2f} percentage points")
ax.set_title("Fold-wise accuracy difference between proposed Model B and baseline")
ax.set_xlabel("Outer fold")
ax.set_ylabel("Accuracy difference (B − A)")
percent_axis(ax)
ax.legend(frameon=False)
# Layout handled globally by constrained_layout.
save_figure(fig, "cv", "Fig_17_Accuracy_Difference_B_Minus_A", "Accuracy difference: proposed minus baseline")

# Figure 18: epochs run in each fold
fig, ax = plt.subplots(figsize=(9.2, 5.0))
for key in DISPLAY_ORDER:
    data = fold_results[fold_results["Model Key"] == key].sort_values("Fold")
    ax.plot(data["Fold"], data["Epochs Run"], marker="o", color=COLORS[key], label=SHORT_LABELS[key])
ax.set_xticks(range(1, 11))
ax.set_title("Epochs completed before early stopping in each fold")
ax.set_xlabel("Outer fold")
ax.set_ylabel("Epochs run")
ax.legend(frameon=False, ncol=2)
# Layout handled globally by constrained_layout.
save_figure(fig, "cv", "Fig_18_Epochs_Run_Across_Folds", "Epochs completed across folds")

# Figure 19: mean training/validation accuracy trajectory across folds
def plot_mean_epoch_trajectory(metric, title, filename):
    fig, ax = plt.subplots(figsize=(9.2, 5.2))
    for key in DISPLAY_ORDER:
        data = epoch_timings[epoch_timings["Model Key"] == key]
        grouped = data.groupby("Epoch")[metric].agg(["mean", "std", "count"]).reset_index()
        grouped["std"] = grouped["std"].fillna(0)
        x_values = grouped["Epoch"].to_numpy(dtype=float)
        mean_values = grouped["mean"].to_numpy(dtype=float)
        sd_values = grouped["std"].to_numpy(dtype=float)
        ax.plot(x_values, mean_values, color=COLORS[key],
                linewidth=2.4 if key == "B" else 1.5, label=SHORT_LABELS[key])
        ax.fill_between(x_values, mean_values - sd_values, mean_values + sd_values,
                        color=COLORS[key], alpha=0.08)
    if "accuracy" in metric:
        percent_axis(ax)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.legend(frameon=False, ncol=2)
    # Layout handled globally by constrained_layout.
    save_figure(fig, "cv", filename, title)

plot_mean_epoch_trajectory("val_accuracy", "Mean validation-accuracy trajectory across the ten folds",
                           "Fig_19_Mean_Validation_Accuracy_Trajectory")
plot_mean_epoch_trajectory("val_loss", "Mean validation-loss trajectory across the ten folds",
                           "Fig_20_Mean_Validation_Loss_Trajectory")




## 3. Final untouched-test results



In [4]:

# Figure 21: final accuracy ranking
final_sorted = final_results.sort_values("Accuracy")
fig, ax = plt.subplots(figsize=(8.5, 5.0))
keys_sorted = final_sorted["Model Key"].astype(str).tolist()
bars = ax.barh(
    [SHORT_LABELS[k] for k in keys_sorted],
    final_sorted["Accuracy"],
    color=[COLORS[k] for k in keys_sorted],
)
ax.bar_label(bars, labels=[f"{100 * value:.2f}%" for value in final_sorted["Accuracy"]], padding=4)
ax.set_xlim(max(0, final_sorted["Accuracy"].min() - 0.04), final_sorted["Accuracy"].max() + 0.015)
percent_axis(ax, "x")
ax.set_title("Final performance on the untouched test cohort")
ax.set_xlabel("Accuracy")
ax.set_ylabel("")
# Layout handled globally by constrained_layout.
save_figure(fig, "final", "Fig_21_Final_Test_Accuracy_Ranking", "Final untouched-test accuracy")

# Figure 22: all final-test metrics
final_long = final_results.melt(
    id_vars=["Model Key"], value_vars=METRICS, var_name="Metric", value_name="Score"
)
fig, ax = plt.subplots(figsize=(10.0, 5.7))
x = np.arange(len(METRICS)); width = 0.19
for index, key in enumerate(DISPLAY_ORDER):
    values = final_long[final_long["Model Key"] == key].set_index("Metric").reindex(METRICS)["Score"]
    ax.bar(x + (index - 1.5) * width, values, width, color=COLORS[key], label=SHORT_LABELS[key])
ax.set_xticks(x)
ax.set_xticklabels(METRICS, rotation=15, ha="right")
ax.set_ylim(0.80, 0.94)
percent_axis(ax)
ax.set_title("All final untouched-test metrics")
ax.set_xlabel("Evaluation metric")
ax.set_ylabel("Score")
ax.legend(frameon=False, ncol=2)
# Layout handled globally by constrained_layout.
save_figure(fig, "final", "Fig_22_Final_Test_All_Metrics", "Final-test metrics comparison")

# Figure 23: CV mean vs final accuracy
cv_accuracy = fold_results.groupby("Model Key")["Accuracy"].agg(["mean", "std"])
comparison_rows = []
for key in DISPLAY_ORDER:
    comparison_rows.extend([
        {"Model Key": key, "Evaluation": "10-fold mean", "Accuracy": cv_accuracy.loc[key, "mean"]},
        {"Model Key": key, "Evaluation": "Final untouched test", "Accuracy": float(final_results.loc[final_results["Model Key"] == key, "Accuracy"].iloc[0])},
    ])
cv_final = pd.DataFrame(comparison_rows)
fig, ax = plt.subplots(figsize=(9.2, 5.2))
sns.barplot(data=cv_final, x="Model Key", y="Accuracy", hue="Evaluation",
            order=DISPLAY_ORDER, palette=["#9ECAE1", "#08519C"], ax=ax)
ax.set_xticks(np.arange(len(DISPLAY_ORDER)))
ax.set_xticklabels([SHORT_LABELS[k] for k in DISPLAY_ORDER], rotation=12, ha="right")
percent_axis(ax)
ax.set_title("Ten-fold mean accuracy versus final untouched-test accuracy")
ax.set_xlabel("")
ax.set_ylabel("Accuracy")
ax.legend(title="Evaluation", frameon=False)
# Layout handled globally by constrained_layout.
save_figure(fig, "final", "Fig_23_CV_Mean_vs_Final_Test_Accuracy", "CV and final accuracy comparison")




## 4. Final-model learning curves



In [5]:

def locate_first(candidates):
    return next((path for path in candidates if path.exists()), None)


for key in DISPLAY_ORDER:
    model_directory = FINAL_MODEL_DIR / f"Model_{key}"
    history_path = locate_first([
        model_directory / "Final_Training_History.csv",
        model_directory / f"Final_Model_{key}_Training_History.csv",
        model_directory / "training_history.csv",
    ])
    if history_path is None:
        skipped_figures.append(f"Final learning curves for Model {key}: history CSV not found in {model_directory}")
        continue

    history = pd.read_csv(history_path)
    epoch_values = history["epoch"] if "epoch" in history.columns else np.arange(1, len(history) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.5))
    if {"accuracy", "val_accuracy"}.issubset(history.columns):
        axes[0].plot(epoch_values, history["accuracy"], label="Training", color="#0072B2", linewidth=2)
        axes[0].plot(epoch_values, history["val_accuracy"], label="Validation", color="#D55E00", linewidth=2)
        axes[0].set_title("Accuracy")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Accuracy")
        percent_axis(axes[0])
        axes[0].legend(frameon=False)
    else:
        axes[0].text(0.5, 0.5, "Accuracy columns unavailable", ha="center", va="center")
    if {"loss", "val_loss"}.issubset(history.columns):
        axes[1].plot(epoch_values, history["loss"], label="Training", color="#0072B2", linewidth=2)
        axes[1].plot(epoch_values, history["val_loss"], label="Validation", color="#D55E00", linewidth=2)
        best_epoch = int(np.nanargmin(history["val_loss"].to_numpy()))
        axes[1].axvline(np.asarray(epoch_values)[best_epoch], color="#009E73", linestyle="--",
                        label=f"Minimum val. loss: epoch {np.asarray(epoch_values)[best_epoch]}")
        axes[1].set_title("Loss")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Loss")
        axes[1].legend(frameon=False)
    else:
        axes[1].text(0.5, 0.5, "Loss columns unavailable", ha="center", va="center")
    fig.suptitle(f"Final-model learning curves — {SHORT_LABELS[key]}")
    # Layout handled globally by constrained_layout.
    save_figure(fig, "learning", f"Final_Model_{key}_Learning_Curves",
                f"Final learning curves for Model {key}")




## 5. Final confusion matrices for all models



In [6]:

def clean_confusion_matrix(frame):
    frame = frame.copy()
    unnamed = [column for column in frame.columns if str(column).startswith("Unnamed")]
    if unnamed:
        frame = frame.drop(columns=unnamed)
    # If the first non-numeric column stores row labels, use it as the index.
    if len(frame.columns) and not pd.api.types.is_numeric_dtype(frame.iloc[:, 0]):
        frame = frame.set_index(frame.columns[0])
    frame = frame.apply(pd.to_numeric, errors="coerce").dropna(axis=0, how="all").dropna(axis=1, how="all")
    return frame


confusion_matrices = {}
for key in DISPLAY_ORDER:
    model_directory = FINAL_MODEL_DIR / f"Model_{key}"
    matrix_path = locate_first([
        model_directory / "Final_Confusion_Matrix.csv",
        model_directory / f"Final_Model_{key}_Confusion_Matrix.csv",
        model_directory / "confusion_matrix.csv",
    ])
    if matrix_path is None:
        skipped_figures.append(f"Final confusion matrices for Model {key}: CSV not found in {model_directory}")
        continue
    matrix = clean_confusion_matrix(pd.read_csv(matrix_path))
    if matrix.shape[0] != matrix.shape[1]:
        skipped_figures.append(f"Model {key} confusion matrix is not square: {matrix_path}")
        continue
    labels = CLASS_ORDER[:matrix.shape[0]]
    matrix.index = labels
    matrix.columns = labels
    confusion_matrices[key] = matrix

    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    sns.heatmap(matrix, annot=True, fmt=".0f", cmap="Blues", cbar=False, linewidths=0.4, ax=ax)
    ax.set_title(f"Final-test confusion matrix — {SHORT_LABELS[key]}")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.tick_params(axis="x", rotation=25)
    ax.tick_params(axis="y", rotation=0)
    # Layout handled globally by constrained_layout.
    save_figure(fig, "confusion", f"Final_Model_{key}_Confusion_Matrix_Counts",
                f"Final confusion matrix counts for Model {key}")

    normalized = matrix.div(matrix.sum(axis=1).replace(0, np.nan), axis=0) * 100
    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    sns.heatmap(normalized, annot=True, fmt=".1f", cmap="YlGnBu", vmin=0, vmax=100,
                cbar_kws={"label": "Row-normalized percentage (%)"}, linewidths=0.4, ax=ax)
    ax.set_title(f"Normalized final-test confusion matrix — {SHORT_LABELS[key]}")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.tick_params(axis="x", rotation=25)
    ax.tick_params(axis="y", rotation=0)
    # Layout handled globally by constrained_layout.
    save_figure(fig, "confusion", f"Final_Model_{key}_Confusion_Matrix_Normalized",
                f"Normalized final confusion matrix for Model {key}")

# Combined normalized matrices
if len(confusion_matrices) == 4:
    fig, axes = plt.subplots(2, 2, figsize=(16.0, 14.0))
    for ax, key in zip(axes.flat, DISPLAY_ORDER):
        matrix = confusion_matrices[key]
        normalized = matrix.div(matrix.sum(axis=1).replace(0, np.nan), axis=0) * 100
        sns.heatmap(normalized, annot=True, fmt=".1f", cmap="YlGnBu", vmin=0, vmax=100,
                    cbar=False, linewidths=0.3, ax=ax)
        ax.set_title(SHORT_LABELS[key])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.tick_params(axis="x", rotation=35, labelsize=7)
        ax.tick_params(axis="y", rotation=0, labelsize=7)
    fig.suptitle("Normalized final-test confusion matrices for all models")
    # Layout handled globally by constrained_layout.
    save_figure(fig, "confusion", "Combined_Normalized_Confusion_Matrices_A_to_D",
                "Combined normalized confusion matrices")




## 6. Class-wise final-test performance



In [7]:

for key in DISPLAY_ORDER:
    model_directory = FINAL_MODEL_DIR / f"Model_{key}"
    report_path = locate_first([
        model_directory / "Final_Classification_Report.csv",
        model_directory / f"Final_Model_{key}_Classification_Report.csv",
        model_directory / "classification_report.csv",
    ])
    if report_path is None:
        skipped_figures.append(f"Class-wise metrics for Model {key}: report CSV not found in {model_directory}")
        continue
    report = pd.read_csv(report_path)
    # Support common sklearn export formats.
    if report.columns[0].startswith("Unnamed") or report.columns[0].lower() in {"class", "label", "index"}:
        report = report.rename(columns={report.columns[0]: "Class"})
    elif "Class" not in report.columns:
        report = report.reset_index().rename(columns={"index": "Class"})
    rename = {}
    for column in report.columns:
        lower = str(column).strip().lower()
        if lower == "precision": rename[column] = "Precision"
        if lower == "recall": rename[column] = "Recall"
        if lower in {"f1-score", "f1_score", "f1"}: rename[column] = "F1-score"
    report = report.rename(columns=rename)
    metric_columns = [c for c in ["Precision", "Recall", "F1-score"] if c in report.columns]
    if not metric_columns:
        skipped_figures.append(f"Unrecognized classification-report format: {report_path}")
        continue
    report["Class"] = report["Class"].astype(str)
    report = report[~report["Class"].str.lower().isin(["accuracy", "macro avg", "weighted avg", "micro avg"])]
    report = report.head(len(CLASS_ORDER))
    long_report = report.melt(id_vars="Class", value_vars=metric_columns,
                              var_name="Metric", value_name="Score")
    fig, ax = plt.subplots(figsize=(9.2, 5.3))
    sns.barplot(data=long_report, y="Class", x="Score", hue="Metric",
                palette=["#0072B2", "#E69F00", "#009E73"], ax=ax)
    ax.set_xlim(0, 1)
    percent_axis(ax, "x")
    ax.set_title(f"Class-wise final-test performance — {SHORT_LABELS[key]}")
    ax.set_xlabel("Score")
    ax.set_ylabel("Ultrasound class")
    ax.legend(title="Metric", frameon=False)
    # Layout handled globally by constrained_layout.
    save_figure(fig, "classwise", f"Final_Model_{key}_Classwise_Metrics",
                f"Class-wise metrics for Model {key}")




## 7. Timing, parameters and efficiency



In [8]:

# Figure 24: CV inference time
comp_key = "Model Key" if "Model Key" in computational.columns else "Key"
comp_plot = computational.set_index(comp_key).reindex(DISPLAY_ORDER)
fig, ax = plt.subplots(figsize=(8.5, 5.0))
bars = ax.bar([SHORT_LABELS[k] for k in DISPLAY_ORDER],
              comp_plot["CV Mean Inference (ms/image)"],
              color=[COLORS[k] for k in DISPLAY_ORDER])
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.set_title("Mean inference time during ten-fold cross-validation")
ax.set_xlabel("")
ax.set_ylabel("Inference time (ms/image)")
ax.tick_params(axis="x", rotation=12)
# Layout handled globally by constrained_layout.
save_figure(fig, "efficiency", "Fig_24_CV_Inference_Time", "Cross-validation inference time")

# Figure 25: final inference time
final_ordered = final_results.set_index("Model Key").reindex(DISPLAY_ORDER)
fig, ax = plt.subplots(figsize=(8.5, 5.0))
bars = ax.bar([SHORT_LABELS[k] for k in DISPLAY_ORDER],
              final_ordered["Inference (ms/image)"],
              color=[COLORS[k] for k in DISPLAY_ORDER])
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.set_title("Inference time of final models on the untouched test set")
ax.set_xlabel("")
ax.set_ylabel("Inference time (ms/image)")
ax.tick_params(axis="x", rotation=12)
# Layout handled globally by constrained_layout.
save_figure(fig, "efficiency", "Fig_25_Final_Test_Inference_Time", "Final-test inference time")

# Figure 26: mean epoch time
fig, ax = plt.subplots(figsize=(8.5, 5.0))
bars = ax.bar([SHORT_LABELS[k] for k in DISPLAY_ORDER],
              comp_plot["CV Mean Epoch (s)"], color=[COLORS[k] for k in DISPLAY_ORDER])
ax.bar_label(bars, fmt="%.2f", padding=3)
ax.set_title("Mean epoch time during ten-fold cross-validation")
ax.set_xlabel("")
ax.set_ylabel("Mean epoch time (s)")
ax.tick_params(axis="x", rotation=12)
# Layout handled globally by constrained_layout.
save_figure(fig, "efficiency", "Fig_26_Mean_Epoch_Time", "Mean epoch time")

# Figure 27: total and trainable parameters
arch_key = "Key" if "Key" in architecture.columns else "Model Key"
arch_plot = architecture.set_index(arch_key).reindex(DISPLAY_ORDER)
fig, ax = plt.subplots(figsize=(9.2, 5.2))
x = np.arange(4); width = 0.36
ax.bar(x - width/2, arch_plot["Parameters"] / 1e6, width, label="Total parameters", color="#56B4E9")
ax.bar(x + width/2, arch_plot["Trainable Parameters"] / 1e6, width, label="Trainable parameters", color="#D55E00")
ax.set_xticks(x)
ax.set_xticklabels([SHORT_LABELS[k] for k in DISPLAY_ORDER], rotation=12)
ax.set_title("Model size and trainable parameter comparison")
ax.set_xlabel("")
ax.set_ylabel("Parameters (millions)")
ax.legend(frameon=False)
# Layout handled globally by constrained_layout.
save_figure(fig, "efficiency", "Fig_27_Model_Parameters", "Model parameter comparison")

# Figure 28: accuracy-inference Pareto plot
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for key in DISPLAY_ORDER:
    x_value = float(comp_plot.loc[key, "CV Mean Inference (ms/image)"])
    y_value = float(cv_accuracy.loc[key, "mean"])
    size = float(arch_plot.loc[key, "Trainable Parameters"]) / 900
    ax.scatter(x_value, y_value, s=size, color=COLORS[key], alpha=0.82,
               edgecolor="white", linewidth=1.2)
    ax.annotate(SHORT_LABELS[key], (x_value, y_value), xytext=(6, 5),
                textcoords="offset points", fontsize=9)
percent_axis(ax)
ax.set_title("Accuracy–inference-time trade-off")
ax.set_xlabel("Mean inference time (ms/image)")
ax.set_ylabel("Ten-fold mean accuracy")
# Layout handled globally by constrained_layout.
save_figure(fig, "efficiency", "Fig_28_Accuracy_Inference_Tradeoff", "Accuracy and inference-time trade-off")

# Figure 29: accuracy versus trainable parameters
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for key in DISPLAY_ORDER:
    x_value = float(arch_plot.loc[key, "Trainable Parameters"]) / 1e6
    y_value = float(cv_accuracy.loc[key, "mean"])
    ax.scatter(x_value, y_value, s=120, color=COLORS[key])
    ax.annotate(SHORT_LABELS[key], (x_value, y_value), xytext=(6, 5), textcoords="offset points")
percent_axis(ax)
ax.set_title("Accuracy relative to trainable model capacity")
ax.set_xlabel("Trainable parameters (millions)")
ax.set_ylabel("Ten-fold mean accuracy")
# Layout handled globally by constrained_layout.
save_figure(fig, "efficiency", "Fig_29_Accuracy_vs_Trainable_Parameters", "Accuracy versus trainable parameters")




## 8. Statistical-result visualizations



In [9]:

if friedman is not None:
    fig, ax = plt.subplots(figsize=(8.4, 5.0))
    data = friedman.sort_values("p-value", ascending=True)
    bars = ax.barh(data["Metric"], data["p-value"],
                   color=["#D55E00" if value < 0.05 else "#999999" for value in data["p-value"]])
    ax.axvline(0.05, color="black", linestyle="--", label="Significance threshold (0.05)")
    ax.set_title("Friedman omnibus tests across Models A–D")
    ax.set_xlabel("p-value")
    ax.set_ylabel("")
    ax.legend(frameon=False)
    # Layout handled globally by constrained_layout.
    save_figure(fig, "statistics", "Fig_30_Friedman_Test_P_Values", "Friedman test p-values")

if holm is not None and {"Comparison", "Metric", "Holm-adjusted p-value"}.issubset(holm.columns):
    holm_matrix = holm.pivot(index="Comparison", columns="Metric", values="Holm-adjusted p-value")
    fig, ax = plt.subplots(figsize=(10.0, 4.3))
    sns.heatmap(holm_matrix, annot=True, fmt=".3f", cmap="YlOrRd_r", vmin=0, vmax=1,
                cbar_kws={"label": "Holm-adjusted p-value"}, linewidths=0.4, ax=ax)
    ax.set_title("Multiplicity-corrected paired comparisons from the original experiment")
    ax.set_xlabel("Metric")
    ax.set_ylabel("Comparison")
    ax.tick_params(axis="x", rotation=20)
    # Layout handled globally by constrained_layout.
    save_figure(fig, "statistics", "Fig_31_Holm_Adjusted_P_Values", "Holm-adjusted comparisons")

# ---------------------------------------------------------------------
# SAVE PUBLICATION TABLES AND MANIFESTS
# ---------------------------------------------------------------------
cv_summary.to_csv(GRAPH_ROOT / "Ten_Fold_Mean_SD_Numeric.csv", index=False)
cv_final.to_csv(GRAPH_ROOT / "CV_Mean_and_Final_Test_Accuracy.csv", index=False)
overlap.to_csv(GRAPH_ROOT / "Patient_Overlap_Matrix.csv")
pd.DataFrame(figure_manifest).to_csv(GRAPH_ROOT / "Figure_Manifest.csv", index=False)
pd.DataFrame({"Skipped or unavailable figure": skipped_figures}).to_csv(
    GRAPH_ROOT / "Skipped_Figures_Report.csv", index=False
)

run_summary = {
    "results_directory": str(RESULTS_DIR),
    "graph_directory": str(GRAPH_ROOT),
    "publication_display_order": DISPLAY_ORDER,
    "proposed_model_internal_key": "B",
    "proposed_model_name": DISPLAY_LABELS["B"],
    "figures_created": len(figure_manifest),
    "skipped_items": len(skipped_figures),
}
with open(GRAPH_ROOT / "Graph_Generation_Summary.json", "w", encoding="utf-8") as file:
    json.dump(run_summary, file, indent=2)

print("\n" + "=" * 100)
print("GRAPH GENERATION COMPLETED")
print("=" * 100)
print(f"Figures created : {len(figure_manifest)}")
print(f"Skipped items   : {len(skipped_figures)}")
print(f"Saved to        : {GRAPH_ROOT}")
if skipped_figures:
    print("\nUnavailable items:")
    for item in skipped_figures:
        print(f" - {item}")
print("\nEvery available figure was saved as PNG (600 dpi), PDF and SVG.")



GRAPH GENERATION COMPLETED
Figures created : 48
Skipped items   : 0
Saved to        : D:\imagecondition revision\MobileNetV1_ABCD_PatientLevel_10Fold_6Class_Results\Publication_Graphs_Model_B_Proposed

Every available figure was saved as PNG (600 dpi), PDF and SVG.
